<a href="https://colab.research.google.com/github/zsgwu/G5_GWU_CAPST/blob/main/Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

# Path to your project folder
project_path = "/content/drive/MyDrive/group-5/RAG_data"

# Change working directory
os.chdir(project_path)

# Verify
print("Current working directory:")
print(os.getcwd())

print("\nFiles in directory:")
print(os.listdir())

Current working directory:
/content/drive/.shortcut-targets-by-id/1HF_VpvP7ySsREVFXjmHKx9Kq2grdYfhc/group-5/RAG_data

Files in directory:
['OES_Report.xlsx', 'Most-Recent-Cohorts-Field-of-Study.csv']


In [5]:
os.makedirs("cleaned", exist_ok=True)
os.makedirs("embeddings", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)

print(os.listdir())

['OES_Report.xlsx', 'Most-Recent-Cohorts-Field-of-Study.csv', 'cleaned', 'embeddings', 'notebooks']


In [9]:
import pandas as pd
import numpy as np

In [11]:
oes = pd.read_excel(
    "OES_Report.xlsx",
    sheet_name=0,
    header=5,          # <-- THIS IS THE KEY FIX
    engine="openpyxl"
)

print(oes.columns)

Index(['Occupation (SOC code)', 'Employment (1)',
       'Annual 25th percentile wage (2)', 'Annual median wage (2)'],
      dtype='object')


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [17]:
# --- Rename columns ---
oes = oes.rename(columns={
    "Occupation (SOC code)": "occupation_raw",
    "Employment (1)": "employment_national",
    "Annual median wage (2)": "annual_median_wage"
})

# --- Drop empty occupation rows ---
oes = oes.dropna(subset=["occupation_raw"])

# --- Extract SOC code ---
oes["soc_code"] = oes["occupation_raw"].str.extract(r"(\d{2}-\d{4})")

# --- Clean occupation title ---
oes["occupation_title"] = (
    oes["occupation_raw"]
    .str.replace(r"\(\d{2}-\d{4}\)", "", regex=True)
    .str.replace("\n", " ")
    .str.strip()
)

# --- Drop rows without SOC ---
oes = oes.dropna(subset=["soc_code"])

# --- Drop aggregate SOC groups ---
oes = oes[
    (oes["soc_code"] != "00-0000") &
    (~oes["soc_code"].str.endswith("0000"))
]

# --- Clean numeric fields ---
for col in ["employment_national", "annual_median_wage"]:
    oes[col] = pd.to_numeric(oes[col], errors="coerce")

# --- Drop rows missing BOTH wage and employment ---
oes = oes.dropna(
    subset=["employment_national", "annual_median_wage"],
    how="all"
)

# --- Add source metadata ---
oes["source"] = "BLS OEWS 2024"

# --- Keep only EduYou columns ---
oes_clean = oes[
    [
        "soc_code",
        "occupation_title",
        "employment_national",
        "annual_median_wage",
        "source"
    ]
].reset_index(drop=True)

# --- Save for RAG ingestion ---
oes_clean.to_csv("cleaned/oes_clean_for_rag.csv", index=False)

print(oes_clean.head(10))
print(f"Final rows: {len(oes_clean)}")

  soc_code                                   occupation_title  \
0  11-1000                                     Top Executives   
1  11-1011                                   Chief Executives   
2  11-1021                    General and Operations Managers   
3  11-1031                                        Legislators   
4  11-2000  Advertising, Marketing, Promotions, Public Rel...   
5  11-2011                Advertising and Promotions Managers   
6  11-2020                       Marketing and Sales Managers   
7  11-2021                                 Marketing Managers   
8  11-2022                                     Sales Managers   
9  11-2030          Public Relations and Fundraising Managers   

   employment_national  annual_median_wage         source  
0            3822780.0            104990.0  BLS OEWS 2024  
1             211850.0            206420.0  BLS OEWS 2024  
2            3584420.0            102950.0  BLS OEWS 2024  
3              26510.0             44810.0  

In [18]:
oes_clean.sample(5)

,soc_code,occupation_title,employment_national,annual_median_wage,source
845,49-3092,Recreational Vehicle Service Technicians,18710.0,50540.0,BLS OEWS 2024
646,41-9000,Other Sales and Related Workers,537090.0,49720.0,BLS OEWS 2024
792,47-4061,Rail-Track Laying and Maintenance Equipment Op...,16480.0,67370.0,BLS OEWS 2024
85,13-2081,"Tax Examiners and Collectors, and Revenue Agents",53530.0,59740.0,BLS OEWS 2024
1011,51-9195,"Molders, Shapers, and Casters, Except Metal an...",34750.0,45690.0,BLS OEWS 2024


In [20]:
# --- Load College Scorecard data ---
scorecard = pd.read_csv(
    "Most-Recent-Cohorts-Field-of-Study.csv",
    low_memory=False
)

# --- Keep ONLY EduYou-relevant columns ---
scorecard = scorecard[
    [
        "CIPCODE",
        "CIPDESC",
        "CREDDESC",
        "INSTNM",
        "CONTROL",
        "EARN_MDN_4YR_NAT"
    ]
]

# --- Rename columns to stable schema ---
scorecard = scorecard.rename(columns={
    "CIPCODE": "cip_code",
    "CIPDESC": "cip_title",
    "CREDDESC": "degree_level",
    "INSTNM": "institution_name",
    "EARN_MDN_4YR_NAT": "median_earnings_4yr_nat"
})

# --- Clean earnings (drop suppressed / missing values) ---
scorecard["median_earnings_4yr_nat"] = pd.to_numeric(
    scorecard["median_earnings_4yr_nat"],
    errors="coerce"
)

scorecard = scorecard.dropna(subset=["median_earnings_4yr_nat"])

# --- Normalize CIP code format (e.g., 110701 → 11.0701) ---
scorecard["cip_code"] = (
    scorecard["cip_code"]
    .astype(str)
    .str.zfill(6)
    .str[:2] + "." +
    scorecard["cip_code"].astype(str).str.zfill(6).str[2:]
)

# --- Filter to degree levels used in EduYou ---
allowed_degrees = [
    "Bachelor's Degree",
    "Master's Degree"
]

scorecard = scorecard[
    scorecard["degree_level"].isin(allowed_degrees)
]

# --- Add source metadata ---
scorecard["source"] = "College Scorecard"

# --- Final EduYou schema ---
scorecard_clean = scorecard[
    [
        "cip_code",
        "cip_title",
        "degree_level",
        "institution_name",
        "CONTROL",
        "median_earnings_4yr_nat",
        "source"
    ]
].reset_index(drop=True)

# --- Save cleaned file for RAG ingestion ---
scorecard_clean.to_csv(
    "cleaned/college_scorecard_clean_for_rag.csv",
    index=False
)

print(scorecard_clean.head())
print(f"Final rows: {len(scorecard_clean)}")

  cip_code                                          cip_title  \
0  00.0109                                   Animal Sciences.   
1  00.0110                       Food Science and Technology.   
2  00.0110                       Food Science and Technology.   
3  00.0199  Agricultural/Animal/Plant/Veterinary Science a...   
4  00.0199  Agricultural/Animal/Plant/Veterinary Science a...   

        degree_level          institution_name CONTROL  \
0  Bachelor's Degree  Alabama A & M University  Public   
1  Bachelor's Degree  Alabama A & M University  Public   
2    Master's Degree  Alabama A & M University  Public   
3  Bachelor's Degree  Alabama A & M University  Public   
4    Master's Degree  Alabama A & M University  Public   

   median_earnings_4yr_nat             source  
0                  49634.0  College Scorecard  
1                  70873.0  College Scorecard  
2                  88332.0  College Scorecard  
3                  65543.0  College Scorecard  
4                  5

In [21]:
# Load CIP–SOC crosswalk
crosswalk = pd.read_excel(
    "CIP2020_SOC2018_Crosswalk.xlsx",
    engine="openpyxl"
)

print(crosswalk.head())
print(crosswalk.columns)

       File Name                                        Description
0        CIP-SOC  This file crosswalks 2020 CIP Codes to 2018 SO...
1        SOC-CIP  This file crosswalks 2018 SOC Codes to 2020 CI...
2        New CIP  This file contains only NEW 2020 CIP Codes. It...
3        New SOC  This file contains only NEW 2018 SOC Codes. It...
4  Added Matches  This file contains only NEW matches that were ...
Index(['File Name', 'Description'], dtype='object')


In [ ]:
crosswalk = crosswalk[
    [
        "CIP2020Code",
        "CIP2020Title",
        "SOC2018Code",
        "SOC2018Title"
    ]
]

In [ ]:
crosswalk = crosswalk.rename(columns={
    "CIP2020Code": "cip_code",
    "CIP2020Title": "cip_title",
    "SOC2018Code": "soc_code",
    "SOC2018Title": "soc_title"
})

# --- Normalize CIP format (110701 → 11.0701) ---
crosswalk["cip_code"] = (
    crosswalk["cip_code"]
    .astype(str)
    .str.zfill(6)
    .str[:2] + "." +
    crosswalk["cip_code"].astype(str).str.zfill(6).str[2:]
)

# --- Normalize SOC format ---
crosswalk["soc_code"] = crosswalk["soc_code"].astype(str)

In [ ]:
# Drop rows missing either side of the mapping
crosswalk = crosswalk.dropna(subset=["cip_code", "soc_code"])

# Drop aggregate SOCs (keep detailed occupations only)
crosswalk = crosswalk[
    (~crosswalk["soc_code"].str.endswith("0000")) &
    (crosswalk["soc_code"] != "00-0000")
]

In [ ]:
crosswalk = crosswalk.drop_duplicates(
    subset=["cip_code", "soc_code"]
).reset_index(drop=True)

In [ ]:
crosswalk.to_csv(
    "cleaned/cip_soc_crosswalk_clean.csv",
    index=False
)

print(crosswalk.head())
print(f"Final crosswalk rows: {len(crosswalk)}")